[Reference](https://medium.com/@Rohan_Dutt/10-forecasting-techniques-that-look-statistically-sound-but-collapse-in-real-world-demand-planning-871a2cde2510)

# 1. Static Model Decay (The “Set-and-Forget” Disaster)

In [1]:
import numpy as np
from sklearn.metrics import mean_squared_error

# 1. Simulate Model Performance over 12 months
# The model is "trained" on data where demand is around 100
months = np.arange(1, 13)
actual_demand = [102, 98, 105, 100, 110, 125, 140, 155, 170, 185, 200, 210] # Market shifts up
fixed_forecast = [100] * 12 # Model never re-trained
# 2. Monitor Mean Squared Error (MSE) on a rolling 3-month window
def monitor_drift(actuals, preds, threshold=500):
    errors = []
    for i in range(3, len(actuals) + 1):
        window_mse = mean_squared_error(actuals[i-3:i], preds[i-3:i])
        errors.append(window_mse)

        if window_mse > threshold:
            print(f"🚨 ALERT: Month {i} - Model Drift Detected (MSE: {window_mse:.2f}). RETRAIN IMMEDIATELY.")
        else:
            print(f"Month {i}: MSE {window_mse:.2f} (Within limits)")
monitor_drift(actual_demand, fixed_forecast)

Month 3: MSE 11.00 (Within limits)
Month 4: MSE 9.67 (Within limits)
Month 5: MSE 41.67 (Within limits)
Month 6: MSE 241.67 (Within limits)
🚨 ALERT: Month 7 - Model Drift Detected (MSE: 775.00). RETRAIN IMMEDIATELY.
🚨 ALERT: Month 8 - Model Drift Detected (MSE: 1750.00). RETRAIN IMMEDIATELY.
🚨 ALERT: Month 9 - Model Drift Detected (MSE: 3175.00). RETRAIN IMMEDIATELY.
🚨 ALERT: Month 10 - Model Drift Detected (MSE: 5050.00). RETRAIN IMMEDIATELY.
🚨 ALERT: Month 11 - Model Drift Detected (MSE: 7375.00). RETRAIN IMMEDIATELY.
🚨 ALERT: Month 12 - Model Drift Detected (MSE: 9775.00). RETRAIN IMMEDIATELY.


# 2. Incentivized Pipeline Bias (The CRM “Oracle” Fallacy)

In [2]:
import pandas as pd

# 1. CRM Data: Reps are optimistic
crm_data = {
    'Deal_ID': ['D1', 'D2', 'D3', 'D4', 'D5'],
    'Value': [100000, 50000, 200000, 80000, 150000],
    'Stage': ['Negotiation', 'Discovery', 'Negotiation', 'Proposal', 'Discovery'],
    'Rep_Prob': [0.90, 0.50, 0.90, 0.70, 0.50] # Rep "Gut Feeling"
}
df = pd.DataFrame(crm_data)
# 2. Historical Reality: Analytics shows actual close rates by stage
# (e.g., only 60% of 'Negotiation' deals actually close)
reality_weights = {
    'Discovery': 0.15,
    'Proposal': 0.40,
    'Negotiation': 0.60
}
# 3. Calculate "Optimism Gap"
df['Stat_Weight'] = df['Stage'].map(reality_weights)
df['Rep_Forecast'] = df['Value'] * df['Rep_Prob']
df['Reality_Forecast'] = df['Value'] * df['Stat_Weight']
print(f"Total Sales-Led Forecast: ${df['Rep_Forecast'].sum():,.2f}")
print(f"Total Reality-Adjusted Forecast: ${df['Reality_Forecast'].sum():,.2f}")
print(f"Risk Exposure (Inflated Pipeline): ${df['Rep_Forecast'].sum() - df['Reality_Forecast'].sum():,.2f}")

Total Sales-Led Forecast: $426,000.00
Total Reality-Adjusted Forecast: $242,000.00
Risk Exposure (Inflated Pipeline): $184,000.00


# 3. Ignoring Cannibalization (The Substitution Blindspot)

In [3]:
import numpy as np
import statsmodels.api as sm

# 1. Simulate Product A (Legacy) and Product B (New)
# Product A demand decreases as Product B sales increase
np.random.seed(42)
price_A = np.random.uniform(10, 12, 50)
price_B = np.random.uniform(8, 10, 50) # Competitive pricing for the new product
# Demand for A influenced by its own price AND the price of B
# A negative coefficient for Price_B would imply complementarity;
# A positive one implies they are substitutes (Cannibalization)
demand_A = 500 - (20 * price_A) + (15 * price_B) + np.random.normal(0, 5, 50)
# 2. Regression to find the "Cannibalization Coefficient"
X = sm.add_constant(pd.DataFrame({'Price_A': price_A, 'Price_B': price_B}))
model = sm.OLS(demand_A, X).fit()
print(model.summary2().tables[1])
cannibal_coeff = model.params['Price_B']
print(f"\nCannibalization Impact: For every $1 decrease in Product B's price,")
print(f"Product A loses {cannibal_coeff:.2f} units of demand.")

              Coef.   Std.Err.          t         P>|t|      [0.025  \
const    540.071823  14.523990  37.184811  1.646306e-36  510.853324   
Price_A  -23.323481   1.085245 -21.491450  5.922650e-26  -25.506712   
Price_B   14.579806   1.021729  14.269736  1.056492e-18   12.524352   

             0.975]  
const    569.290323  
Price_A  -21.140250  
Price_B   16.635261  

Cannibalization Impact: For every $1 decrease in Product B's price,
Product A loses 14.58 units of demand.


# 4. Cognitive Bias in Model Overrides (The “Expert” Trap)

In [4]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error as mape

# 1. Historical Data
data = {
    'Month': ['Jan', 'Feb', 'Mar', 'Apr', 'May'],
    'Actual_Sales': [100, 110, 105, 120, 115],
    'Stat_Forecast': [102, 108, 107, 118, 112],
    'Human_Adjusted': [105, 115, 120, 130, 125] # Over-optimistic overrides
}
df = pd.DataFrame(data)
# 2. Calculate MAPE for both
stat_mape = mape(df['Actual_Sales'], df['Stat_Forecast'])
human_mape = mape(df['Actual_Sales'], df['Human_Adjusted'])
# 3. Calculate FVA (Positive means human helped, Negative means human hurt)
fva = stat_mape - human_mape
print(f"Statistical Forecast Error: {stat_mape:.2%}")
print(f"Human Adjusted Error: {human_mape:.2%}")
print(f"Forecast Value Added (FVA): {fva:.2%}")
if fva < 0:
    print("Result: Human intervention reduced forecast accuracy.")
else:
    print("Result: Human intervention improved forecast accuracy.")

Statistical Forecast Error: 2.00%
Human Adjusted Error: 8.17%
Forecast Value Added (FVA): -6.17%
Result: Human intervention reduced forecast accuracy.


# 5. Naive Seasonality Adjustments (The “Last Year” Mirror)

In [5]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose

# 1. Simulate 2 years of demand with a peak in Month 12
months = np.arange(24)
seasonal_pattern = np.array([0.8, 0.8, 0.9, 1.0, 1.1, 1.2, 1.1, 1.0, 0.9, 1.0, 1.3, 1.8] * 2)
demand = 100 * seasonal_pattern + np.random.normal(0, 5, 24)
# 2. Simulate a "Real-Time Signal" (e.g., Google Trends for Month 24 shows a 20% drop)
real_time_signal = np.ones(24)
real_time_signal[-1] = 0.8 # Consumers aren't searching for the product this year
# 3. Apply Naive Forecast vs. Adjusted Forecast
df = pd.DataFrame({'Month': months, 'Demand': demand, 'Trend_Signal': real_time_signal})
df['Naive_Forecast'] = df['Demand'].shift(12) # Just use last year's actuals
# Adjustment Logic: Blend History (70%) with Real-Time Signal (30%)
df['Adjusted_Forecast'] = (df['Naive_Forecast'] * 0.7) + ((df['Naive_Forecast'] * df['Trend_Signal']) * 0.3)
print("--- Month 24 Forecast Comparison ---")
print(f"Naive Forecast (Last Year): {df['Naive_Forecast'].iloc[-1]:.2f}")
print(f"Adjusted Forecast (Current Sentiment): {df['Adjusted_Forecast'].iloc[-1]:.2f}")

--- Month 24 Forecast Comparison ---
Naive Forecast (Last Year): 178.39
Adjusted Forecast (Current Sentiment): 167.69


# 6. Deterministic Lead Time Assumptions (The “Static Supply” Myth)

In [6]:
import numpy as np
import scipy.stats as stats

# 1. Setup: Daily Demand is stable
avg_daily_demand = 100
std_demand = 10
# 2. Scenario A: Fixed Lead Time (The "Naive" Way)
fixed_lead_time = 20  # Days
naive_rop = (avg_daily_demand * fixed_lead_time) + (1.65 * std_demand * np.sqrt(fixed_lead_time))
# 3. Scenario B: Variable Lead Time (The "Real-World" Way)
# Lead time follows a Gamma distribution (min 15 days, mean 20, occasional 40+ day delays)
np.random.seed(42)
lt_samples = np.random.gamma(shape=20, scale=1, size=10000)
# Reorder Point formula accounting for BOTH Demand and Lead Time Variance
# Formula: (D_avg * LT_avg) + Z * sqrt( (LT_avg * sigma_D^2) + (D_avg^2 * sigma_LT^2) )
lt_avg = np.mean(lt_samples)
lt_std = np.std(lt_samples)
z_score = 1.65 # 95% Service Level
robust_rop = (avg_daily_demand * lt_avg) + z_score * np.sqrt(
    (lt_avg * (std_demand**2)) + ((avg_daily_demand**2) * (lt_std**2))
)
print(f"Naive Reorder Point (Fixed LT): {naive_rop:.0f} units")
print(f"Robust Reorder Point (Variable LT): {robust_rop:.0f} units")
print(f"Inventory Buffer Gap: {robust_rop - naive_rop:.0f} units (Risk of Stockout if ignored)")

Naive Reorder Point (Fixed LT): 2074 units
Robust Reorder Point (Variable LT): 2745 units
Inventory Buffer Gap: 671 units (Risk of Stockout if ignored)


# 7. Gaussian Distribution Assumptions (The Bell Curve Blindspot)

In [7]:
import numpy as np

# 1. Setup: Avg demand of 500 units, Std Dev of 50
mu, sigma = 500, 50
confidence_level = 0.99  # Standard "high" confidence
# 2. Traditional Gaussian Safety Stock (Z-score for 99% = 2.33)
theoretical_safety_stock = 2.33 * sigma
print(f"Gaussian Safety Stock: {theoretical_safety_stock:.2f} units")
# 3. Monte Carlo: Simulate demand with "Fat Tails" (Student's T distribution)
# This simulates a world where Black Swans (COVID, Strikes) happen
simulated_demand = mu + np.random.standard_t(df=3, size=10000) * sigma
# Calculate the actual 99th percentile from simulation
actual_99_percentile = np.percentile(simulated_demand, 99)
required_safety_stock = actual_99_percentile - mu
print(f"Real-World Required Safety Stock: {required_safety_stock:.2f} units")
print(f"Shortfall using 'Standard' Math: {required_safety_stock - theoretical_safety_stock:.2f} units")

Gaussian Safety Stock: 116.50 units
Real-World Required Safety Stock: 218.26 units
Shortfall using 'Standard' Math: 101.76 units


# 8. High R-Squared as a Success Metric (The Overfitting Trap)

In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Historical Data (Day 1-20) + Future Unseen Data (Day 21-25)
days = np.arange(1, 26).reshape(-1, 1)
history_y = 100 + 2*days[:20] + np.random.normal(0, 3, 20).reshape(-1,1)
future_y = 100 + 2*days[20:] + np.random.normal(0, 3, 5).reshape(-1,1)
# 2. Overfit Model: 15th Degree Polynomial
poly = PolynomialFeatures(degree=15)
X_poly = poly.fit_transform(days[:20])
model = LinearRegression().fit(X_poly, history_y)
# 3. Evaluate: Perfect Training vs. Disastrous Testing
train_pred = model.predict(X_poly)
test_pred = model.predict(poly.transform(days[20:]))
print(f"Historical R-Squared: {r2_score(history_y, train_pred):.4f} (Looks Perfect!)")
print(f"Future Forecast Error (MAE): {mean_absolute_error(future_y, test_pred):.2f}")
# The MAE will be massive compared to the training error

Historical R-Squared: 0.9614 (Looks Perfect!)
Future Forecast Error (MAE): 67650.15


# 9. Moving Averages (The “Lagging Indicator” Fallacy)


In [10]:
import pandas as pd
import numpy as np


# 1. Simulate a sudden trend shift (e.g., product going viral at Day 20)
np.random.seed(42)
days = np.arange(50)
base_demand = 100 + np.random.normal(0, 5, 50)
trend_spike = np.where(days > 20, (days - 20) * 15, 0)
demand = base_demand + trend_spike
df = pd.DataFrame({'Day': days, 'Actual': demand})
# 2. Compare 10-day SMA vs 10-day EWMA
# SMA: Equal weight | EWMA: Exponentially decaying weight
df['SMA_10'] = df['Actual'].rolling(window=10).mean()
df['EWMA_10'] = df['Actual'].ewm(span=10, adjust=False).mean()
# 3. Calculate "Lag Error" (Mean Absolute Error during the trend phase)
trend_phase = df.iloc[21:]
sma_mae = (trend_phase['Actual'] - trend_phase['SMA_10']).abs().mean()
ewma_mae = (trend_phase['Actual'] - trend_phase['EWMA_10']).abs().mean()
print(f"SMA Lag-Induced Error (MAE): {sma_mae:.2f}")
print(f"EWMA Response Error (MAE): {ewma_mae:.2f}")
print(f"Accuracy Improvement: {((sma_mae - ewma_mae) / sma_mae)*100:.1f}%")

SMA Lag-Induced Error (MAE): 61.29
EWMA Response Error (MAE): 56.97
Accuracy Improvement: 7.0%


# 10. Linear Regression (The “Straight-Line” Trap)

In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_absolute_error

# 1. Simulate Volatile Demand (Stable trend + sudden TikTok spike)
np.random.seed(42)
time = np.arange(30).reshape(-1, 1)
demand = 100 + (2.5 * time.flatten()) + np.random.normal(0, 5, 30)
demand[25] = 450  # The "Black Swan" event (outlier)

df = pd.DataFrame({'Day': time.flatten(), 'Actual_Demand': demand})

def adaptive_forecasting(data, threshold=2.0):
    """
    Detects if Linear Regression error is too high and
    switches to a Robust Estimator.
    """
    X = data[['Day']]
    y = data['Actual_Demand']

    # Fit standard Linear Regression
    lr = LinearRegression().fit(X, y)
    df['LR_Pred'] = lr.predict(X)

    # Calculate Residuals and Z-Scores
    residuals = y - df['LR_Pred']
    z_scores = np.abs((residuals - residuals.mean()) / residuals.std())

    # Logic: If any recent Z-score exceeds threshold, Linear Regression is compromised
    if z_scores.iloc[-5:].max() > threshold:
        print(f"⚠️ Volatility Detected (Z-Score: {z_scores.max():.2f}). Switching to Huber Regressor.")
        # Huber Regressor is robust to outliers
        robust_model = HuberRegressor().fit(X, y)
        return robust_model.predict(X), "Robust/Adaptive"
    else:
        return df['LR_Pred'], "Standard Linear"

# Execute
predictions, model_type = adaptive_forecasting(df)
df['Final_Forecast'] = predictions

print(f"Model Strategy Used: {model_type}")
print(f"Final Day Forecast: {df['Final_Forecast'].iloc[-1]:.2f}")

⚠️ Volatility Detected (Z-Score: 5.14). Switching to Huber Regressor.
Model Strategy Used: Robust/Adaptive
Final Day Forecast: 169.21
